In [ ]:
!pip install -q mediapipe kagglehub pandas numpy tqdm opencv-python

import os
import cv2
import urllib.request
import numpy as np
import pandas as pd
from tqdm import tqdm
import kagglehub
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Unduh model MediaPipe Hand Landmarker
MODEL_PATH = "hand_landmarker.task"
if not os.path.exists(MODEL_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, MODEL_PATH)
    print("Model MediaPipe berhasil diunduh.")

# 2. Unduh Dataset Kandidat 2 (Kaggle Unimal)
dataset_path = kagglehub.dataset_download("agungmrf/indonesian-sign-language-bisindo")
print("Dataset berhasil diunduh di:", dataset_path)

Model MediaPipe berhasil diunduh.
Using Colab cache for faster access to the 'indonesian-sign-language-bisindo' dataset.
Dataset berhasil diunduh di: /kaggle/input/indonesian-sign-language-bisindo


In [ ]:
def normalize_single_hand(landmarks):
    """Normalisasi 21 titik (x, y, z) relatif terhadap wrist (titik 0) dan skala ukuran tangan."""
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32)

    # 1. Translasi: titik 0 (wrist) menjadi pusat (0, 0, 0)
    wrist = coords[0].copy()
    coords = coords - wrist

    # 2. Scaling: bagi dengan jarak Euclidean maksimum dari wrist agar kebal jarak kamera
    max_dist = np.max(np.linalg.norm(coords, axis=1))
    if max_dist > 0:
        coords = coords / max_dist

    return coords.flatten()  # 63 fitur (21 x 3)

def extract_feature_vector(detection_result):
    """Menghasilkan vektor 126 fitur yang konsisten untuk 1 maupun 2 tangan."""
    if not detection_result.hand_landmarks:
        return None

    # Urutkan tangan dari kiri ke kanan berdasarkan koordinat X wrist di layar agar urutannya konsisten
    hands_sorted = sorted(detection_result.hand_landmarks, key=lambda h: h[0].x)

    # Tangan pertama (63 fitur)
    hand_1_features = normalize_single_hand(hands_sorted[0])

    # Tangan kedua (63 fitur jika ada, atau 63 angka nol jika 1 tangan)
    if len(hands_sorted) > 1:
        hand_2_features = normalize_single_hand(hands_sorted[1])
    else:
        hand_2_features = np.zeros(63, dtype=np.float32)

    return np.concatenate([hand_1_features, hand_2_features])  # Total: 126 fitur

In [ ]:
from collections import defaultdict

# 1. Kumpulkan semua file gambar dari seluruh subfolder A-Z (mencakup train & val)
class_images = defaultdict(list)
for root, _, files in os.walk(dataset_path):
    folder_name = os.path.basename(root).strip().upper()
    if len(folder_name) == 1 and folder_name.isalpha():
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                class_images[folder_name].append(os.path.join(root, f))

classes = sorted(class_images.keys())
total_imgs = sum(len(v) for v in class_images.values())
print(f"Ditemukan {len(classes)} kelas dengan total {total_imgs} gambar.")

# 2. Inisialisasi MediaPipe sesuai parameter audit (num_hands=2, confidence=0.5)
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

features_list = []
labels_list = []
hand_counts = []

# 3. Proses ekstraksi fitur per kelas
for label in classes:
    image_paths = class_images[label]

    for img_path in tqdm(image_paths, desc=f"Ekstraksi Kelas {label}", leave=False):
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        result = detector.detect(mp_image)

        vec = extract_feature_vector(result)
        if vec is not None:
            features_list.append(vec)
            labels_list.append(label)
            hand_counts.append(len(result.hand_landmarks))

X = np.array(features_list, dtype=np.float32)
y = np.array(labels_list)

# 4. Simpan dalam format terkompresi (.npz) dan CSV
os.makedirs("artifacts", exist_ok=True)
np.savez_compressed("artifacts/bisindo_unimal_features.npz", X=X, y=y, num_hands=np.array(hand_counts))

feature_cols = [f"h{h}_p{p}_{axis}" for h in (1, 2) for p in range(21) for axis in ('x', 'y', 'z')]
df_features = pd.DataFrame(X, columns=feature_cols)
df_features.insert(0, "label", y)
df_features.insert(1, "detected_hands", hand_counts)
df_features.to_csv("artifacts/bisindo_unimal_features.csv", index=False)

print(f"\nEkstraksi Selesai! Total sampel berhasil: {len(X)} | Dimensi fitur: {X.shape}")

Ditemukan 26 kelas dengan total 11470 gambar.



Ekstraksi Selesai! Total sampel berhasil: 9070 | Dimensi fitur: (9070, 126)


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Muat data fitur hasil ekstraksi (9070, 126)
data = np.load("artifacts/bisindo_unimal_features.npz")
X = data["X"]
y = data["y"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. Bagi data: 80% Train (7.256 sampel) & 20% Test (1.814 sampel) secara merata per kelas
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 3. Latih 2 kandidat model Baseline: Random Forest & MLP (Neural Network)
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=100, random_state=42)
}

best_model = None
best_acc = 0.0
best_name = ""

for name, model in models.items():
    print(f"Melatih {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    print(f"--> {name} | Accuracy: {acc*100:.2f}% | Macro F1-Score: {f1*100:.2f}%\n")

    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_name = name
        best_pred = y_pred

# 4. Tampilkan laporan akurasi per huruf (A-Z) dari model terbaik
print(f"=== LAPORAN KLASIFIKASI MODEL TERBAIK ({best_name}) ===")
print(classification_report(y_test, best_pred, target_names=le.classes_, digits=4))

# 5. Simpan model terbaik dan laporan evaluasi CSV
joblib.dump({"model": best_model, "label_encoder": le}, "artifacts/bisindo_baseline_model.joblib")

report_dict = classification_report(y_test, best_pred, target_names=le.classes_, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose().reset_index().rename(columns={"index": "class"})
df_report.to_csv("artifacts/baseline_classification_report.csv", index=False)
print("Model dan laporan evaluasi berhasil disimpan di folder artifacts/")

Melatih Random Forest...
--> Random Forest | Accuracy: 94.54% | Macro F1-Score: 94.49%

Melatih MLP Neural Net...
--> MLP Neural Net | Accuracy: 95.76% | Macro F1-Score: 95.72%

=== LAPORAN KLASIFIKASI MODEL TERBAIK (MLP Neural Net) ===
              precision    recall  f1-score   support

           A     0.9412    0.9231    0.9320        52
           B     0.9524    0.9091    0.9302        66
           C     0.9167    0.8594    0.8871        64
           D     0.9692    0.8630    0.9130        73
           E     0.9189    0.9855    0.9510        69
           F     0.9836    0.9231    0.9524        65
           G     1.0000    0.9853    0.9926        68
           H     1.0000    0.9846    0.9922        65
           I     1.0000    0.9868    0.9934        76
           J     0.9868    1.0000    0.9934        75
           K     0.9688    0.9688    0.9688        64
           L     0.9315    0.9714    0.9510        70
           M     0.9043    0.9551    0.9290        89
      

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
